In [14]:
import sys
from pathlib import Path

CURRENT_DIR = Path.cwd()
RF_UTILS_DIR = CURRENT_DIR.parent / "randomForest"
sys.path.insert(0, str(RF_UTILS_DIR))

In [15]:
import json
import time
import numpy as np
import os
from pathlib import Path
from xgboost import XGBClassifier

from train_utils import build_train_val_tables
from eval_utils import evaluate, predictions_to_mask, morphological_cleanup
from features import extract_features, load_image_mask_pair
from xgb_hypothesis_config import HYPOTHESES

In [16]:
PROJECT_ROOT = Path.cwd().parent

DATASET_ROOT = Path(
    os.getenv("EWS_DATASET_PATH", PROJECT_ROOT / "data" / "EWS-Dataset")
)

TRAIN_DIR = DATASET_ROOT / "train"
VAL_DIR   = DATASET_ROOT / "validation"
TEST_DIR  = DATASET_ROOT / "test"

X_train, y_train, val_img_paths, val_mask_paths = build_train_val_tables(TRAIN_DIR, VAL_DIR)

Train: 142 images
Val:   24 images
Building training table...
X_train: (1396180, 13)  y_train: (1396180,)


In [17]:
def run_trial(params, train_X, train_y, val_img_paths, val_mask_paths):
    model = XGBClassifier(
        **params,
        n_jobs=-1,
    )

    t0 = time.time()
    model.fit(train_X, train_y)
    train_time = time.time() - t0

    ious = []
    t1 = time.time()

    for img_path, mask_path in zip(val_img_paths, val_mask_paths):
        img_rgb, gt_mask = load_image_mask_pair(img_path, mask_path)
        X = extract_features(img_rgb)
        y_pred = model.predict(X)
        pred_mask = predictions_to_mask(y_pred, 350, 350)
        pred_mask = morphological_cleanup(pred_mask)
        m = evaluate(gt_mask, pred_mask, apply_cleanup=False)
        ious.append(m["iou"])

    inference_time = time.time() - t1

    return {
        "mean_iou": round(float(np.mean(ious)), 6),
        "std_iou": round(float(np.std(ious)), 6),
        "train_time_s": round(train_time, 3),
        "inference_time_s": round(inference_time, 3),
    }

In [18]:
all_results = []

for trial in HYPOTHESES:
    print(f"Running: {trial['hypothesis']} ...")
    metrics = run_trial(
        trial["params"],
        X_train,
        y_train,
        val_img_paths,
        val_mask_paths,
    )

    all_results.append({
        "hypothesis": trial["hypothesis"],
        "rationale": trial["rationale"],
        "params": trial["params"],
        **metrics,
    })

    print(
        f" IoU={metrics['mean_iou']:.4f} ±{metrics['std_iou']:.4f} "
        f"train={metrics['train_time_s']:.1f}s"
    )

print("\nDone.")

Running: baseline ...
 IoU=0.8402 ±0.1293 train=25.5s
Running: shallower_trees ...
 IoU=0.8421 ±0.1281 train=17.1s
Running: lower_lr_more_trees ...
 IoU=0.8395 ±0.1302 train=39.4s
Running: more_row_sampling ...
 IoU=0.8397 ±0.1295 train=21.4s

Done.


In [19]:
best = max(all_results, key=lambda x: x["mean_iou"])

output = {
    "best_hypothesis": best["hypothesis"],
    "best_params": best["params"],
    "best_iou": best["mean_iou"],
    "rationale": best["rationale"],
    "n_trials": len(all_results),
    "all_trials": all_results,
}

with open("xgb_hyperparam_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(f"Best: {best['hypothesis']}  IoU={best['mean_iou']:.4f}")
print("Saved: xgb_hyperparam_results.json")

Best: shallower_trees  IoU=0.8421
Saved: xgb_hyperparam_results.json


In [20]:
all_results = []

for h in HYPOTHESES:
    print(f"\nRunning: {h['hypothesis']}")
    print(h["rationale"])

    result = run_xgb_trial(
        h["params"],
        X_train,
        y_train,
        val_img_paths,
        val_mask_paths,
        apply_cleanup=True,
    )

    row = {
        "hypothesis": h["hypothesis"],
        "rationale": h["rationale"],
        "params": h["params"],
        "mean_iou": round(result["mean_iou"], 6),
        "std_iou": round(result["std_iou"], 6),
        "train_time_s": round(result["train_time_s"], 3),
        "inference_time_s": round(result["inference_time_s"], 3),
    }

    all_results.append(row)

    print(
        f"mean IoU={row['mean_iou']:.4f}, "
        f"std={row['std_iou']:.4f}, "
        f"train={row['train_time_s']:.2f}s, "
        f"infer={row['inference_time_s']:.2f}s"
    )


Running: baseline
Reference XGBoost configuration.
mean IoU=0.8402, std=0.1293, train=16.17s, infer=5.32s

Running: shallower_trees
Reduce overfitting and noisy boundaries.
mean IoU=0.8421, std=0.1281, train=15.16s, infer=4.27s

Running: lower_lr_more_trees
More gradual boosting may generalise better.
mean IoU=0.8395, std=0.1302, train=30.99s, infer=8.27s

Running: more_row_sampling
Increase robustness by stronger stochasticity.
mean IoU=0.8397, std=0.1295, train=16.74s, infer=5.34s
